## XBRL US API - ACFR statements by report  

### Authenticate for access token 
Click the run button (Colab) or in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load credentials (if .json exists)
creds = {}
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        print("Loaded .json")
    except Exception as e:
        print("Warning: failed to read from .json:", e)

if creds:
    # Try nested prod object first
    selected = None
    if isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prod-prefixed keys
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        for k in keys:
            prefixed_key = 'prod' + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prod variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    print('Using credentials from .json as available.')
else:
    # No creds.json — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

The cell below is optional - it creates a list of report ID values and ACFR statement titles that can be used in the query below to filter the data frame.

In [ ]:
# @title
### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED


# Complete list of all ACFRs in the Public Filings Database
report_endpoint = 'report'
params = { 
    'report.source-name': 'GRIP',
    'fields':  'report.entity-name,report.year-focus,report.id,report.entry-url,report.source-name'  
}
search_endpoint = 'https://api.xbrl.us/api/v1/' + report_endpoint + '/search'
report_fields = params['fields']
offset_value = 0
report_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        reports = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        reports_json = reports.json()
        if 'error' in reports_json:
            if reports_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(reports_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + reports_json['paging']['limit']), "records are found so far ...")

    report_df += reports_json['data']

    if reports_json['paging']['count'] < reports_json['paging']['limit']:
        print(" - this set contained fewer than the", reports_json['paging']['limit'], "possible, only", str(reports_json['paging']['count']), "records.")
        break
    else: 
        offset_value += reports_json['paging']['limit'] 
        if 100 == reports_json['paging']['limit']:
                params['fields'] = report_fields + ',' + report_endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * reports_json['paging']['limit']:
                        break 
        elif 500 == reports_json['paging']['limit']:
                params['fields'] = report_fields + ',' + report_endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * reports_json['paging']['limit']:
                        break 
        params['fields'] = report_fields + ',' + report_endpoint + '.offset({})'.format(offset_value)

if not 'error' in reports_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(report_df).index
    total_rows = len(index)
    your_limit = reports_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    report_df = pd.DataFrame(report_df)


# Unique list of report section titles
relationship_endpoint = 'relationship'
params = { 
    'dts.id': '821819,926240',
    'network.link-name': 'presentationLink',
    'unique': '',
    'fields':  'network.role-description'  
}
search_endpoint = 'https://api.xbrl.us/api/v1/' + relationship_endpoint + '/search'
relationship_fields = params['fields']
offset_value = 0
relationship_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        relationships = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        relationships_json = relationships.json()
        if 'error' in relationships_json:
            if relationships_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(relationships_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + relationships_json['paging']['limit']), "records are found so far ...")

    relationship_df += relationships_json['data']

    if relationships_json['paging']['count'] < relationships_json['paging']['limit']:
        print(" - this set contained fewer than the", relationships_json['paging']['limit'], "possible, only", str(relationships_json['paging']['count']), "records.")
        break
    else: 
        offset_value += relationships_json['paging']['limit'] 
        if 100 == relationships_json['paging']['limit']:
                params['fields'] = relationship_fields + ',' + relationship_endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * relationships_json['paging']['limit']:
                        break 
        elif 500 == relationships_json['paging']['limit']:
                params['fields'] = relationship_fields + ',' + relationship_endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * relationships_json['paging']['limit']:
                        break 
        params['fields'] = relationship_fields + ',' + relationship_endpoint + '.offset({})'.format(offset_value)

if not 'error' in relationships_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(relationship_df).index
    total_rows = len(index)
    your_limit = relationships_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    relationships_df = pd.DataFrame(relationship_df)

from IPython.display import display, HTML

# Prepare the report details as a single HTML string
report_details_html = report_df.to_html(index=False, classes='table table-striped', border=0)

# Prepare the statement names as a single HTML string
statement_names_html = relationships_df.to_html(index=False, header=False, classes='table table-striped', border=0)

# Create a responsive HTML table with two columns, each max 40% width, and word-break for long descriptions
responsive_html = f"""
<style>
.responsive-table {{
    display: flex;
    flex-wrap: wrap;
    gap: 20px;
}}
.responsive-table > div {{
    flex: 1 1 0;
    min-width: 300px;
    max-width: 50%;
    box-sizing: border-box;
}}
.table {{
    width: 100%;
    border-collapse: collapse;
    table-layout: fixed;
}}
.table th, .table td {{
    padding: 8px;
    border: 1px solid #ddd;
    word-break: break-word;
    max-width: 40%;
}}
.statement-names-container {{
    word-break: break-word !important;
    max-width: 40%px;
    overflow-wrap: break-word;
}}
</style>
<div class="responsive-table">
    <div>
        <h4>Report Details</h4>
        {report_details_html}
    </div>
    <div>
        <h4>Statement Names</h4>
        <div class="statement-names-container">{statement_names_html}</div>
    </div>
</div>
"""

display(HTML(responsive_html))

### Make a query
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run_** menu option with the cell **immediately below this text** to run the query for updated results.

The sample results are from a set of ACFR reports posted to the XBRL US Public Filings Database.  To test for results quickly, modify the **_report\_ids_** to shorten the list, and change the **_XBRL\_Elements_** to return different data from an ACFR statement.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return.

In [ ]:
# Define the parameters of the query. Run the preceding cell to generate report values and ACFR statement titles that can be added to the list below

endpoint = 'cube' #taxonomy presentation linkbase + facts

report_ids = [ 
'737426', # This id is for the City of Flint - view the report at https://xbrlus.github.io/acfr/ixviewer/ix.html?doc=../samples/119/City-of-Flint-20220630-Annual-Accounts.xhtml
]

Statements = [
		# remove the Statement between the quotes to get all data for report.ids below
		'200110 - Statement - Activities - General Revenues and Changes in Net Position'
		] 

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
		'report.id',
		'dts.id',
		'cube.description.sort(ASC)',
		'cube.tree-sequence.sort(ASC)',
		'report.entity-name',
		'dimension-pair',
		'cube.primary-local-name',
		'fact.value',
		'unit',
		'period.fiscal-year.sort(DESC)'
        ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 10 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = {
         'report.source-name': 'GRIP', 
         #'cube.description': ','.join(Statements),
         #'report.id': ','.join(report_ids), #comment out to get all reports
         'fields': ','.join(fields),
         'unique': ''
         }

print('\n\nNext click the run button (Colab) or in the gray code cell below, then click the Run button above to execute the query for results.\n\n')


In [ ]:
# @title
### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Mon Jun 30 17:53:51 2025 training@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
up to 10000 records are found so far ...
up to 15000 records are found so far ...
up to 20000 records are found so far ...
up to 25000 records are found so far ...
up to 30000 records are found so far ...
up to 35000 records are found so far ...
up to 40000 records are found so far ...
up to 45000 records are found so far ...
up to 50000 records are found so far ...
up to 55000 records are found so far ...
up to 60000 records are found so far ...
up to 65000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 4614 records.

At Mon Jun 30 18:07:56 2025, the query finished with   64614   rows returned in 0:14:04.632502 for 
https://api.xbrl.us/api/v1/cube/search?unique&report.source-name=GRIP&fields=report.id,dts.id,cube.description.sort(ASC),cube.tree-sequence.sort(ASC),report.entity-name,dimension-pair,cube.primary-local-

,report.id,dts.id,cube.description,cube.tree-sequence,report.entity-name,dimension-pair,cube.primary-local-name,fact.value,unit,period.fiscal-year
0,737434,926240,100000 - Statement - Net Position,14,Davison Community Schools (School District),[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],Cash,38190093,USD,2023
1,737433,926240,100000 - Statement - Net Position,14,Davison Community Schools (School District),[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],Cash,29710437,USD,2022
2,723388,926240,100000 - Statement - Net Position,23,Pine River Township,[{'TypeOfGovernmentUnitAxis': 'BusinessTypeActivitiesMember'}],CashAndCashEquivalents,1842537,USD,2023
3,723388,926240,100000 - Statement - Net Position,23,Pine River Township,[{'TypeOfGovernmentUnitAxis': 'GovernmentalActivitiesMember'}],CashAndCashEquivalents,1513675,USD,2023
4,723388,926240,100000 - Statement - Net Position,23,Pine River Township,[{'TypeOfGovernmentUnitAxis': 'PrimaryGovernmentActivitiesMember'}],CashAndCashEquivalents,3356212,USD,2023
...,...,...,...,...,...,...,...,...,...,...
64609,677267,821819,808750 - Form 5572 Michigan - OPEB,14,"Flint, Michigan",[{'OPEBPlanNameAxis': 'OPEB Plan'}],OPEBPlanRetiredEmployeesAndBeneficiaries,1273,pure,2021
64610,677267,821819,808750 - Form 5572 Michigan - OPEB,21,"Flint, Michigan",[{'OPEBPlanNameAxis': 'OPEB Plan'}],OPEBAssumedLongTermRateOfReturn,0.07,pure,2021
64611,677267,821819,808750 - Form 5572 Michigan - OPEB,22,"Flint, Michigan",[{'OPEBPlanNameAxis': 'OPEB Plan'}],OPEBDiscountRate,0.02,pure,2021
64612,677267,821819,808750 - Form 5572 Michigan - OPEB,26,"Flint, Michigan",[{'OPEBPlanNameAxis': 'OPEB Plan'}],OPEBAssumedInflationRateNextYear,0.03,pure,2021


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('data.csv')
#!cp data.csv "drive/My Drive/"